In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
from datetime import datetime, timedelta
SEED = 42
rng = np.random.default_rng(SEED)

Global config

In [3]:
N_USERS = 5000                       # waitlist signups simulated
WAITLIST_START = datetime(2025, 9, 1)  # pre-launch waitlist opens
WAITLIST_DAYS = 90                   # 90-day waitlist period
BETA_START = datetime(2025, 12, 1)   # simulated 8-week early-access beta
BETA_DAYS = 56

# Opinion categories on an opinion-first social network, with relative
# popularity (share of activity) and a base "engagement multiplier".
CATEGORIES = {
    #                 share_weight, engagement_mult
    "Sports":            (0.16, 1.35),
    "Movies & TV":       (0.14, 1.25),
    "Memes & Humour":    (0.13, 1.45),
    "Music":             (0.11, 1.20),
    "Food & Drink":      (0.10, 1.05),
    "Gaming":            (0.09, 1.15),
    "Tech & Gadgets":    (0.08, 1.00),
    "Fashion & Beauty":  (0.06, 0.95),
    "Travel":            (0.05, 0.85),
    "Current Affairs":   (0.04, 0.80),
    "Books":             (0.02, 0.75),
    "Lifestyle":         (0.02, 0.90),
}
cat_names = list(CATEGORIES.keys())
cat_share = np.array([v[0] for v in CATEGORIES.values()])
cat_share = cat_share / cat_share.sum()
cat_eng = {k: v[1] for k, v in CATEGORIES.items()}

# Acquisition channels: share of signups + a funnel-quality multiplier.
# Paid/social_ads users are cheaper but less committed; referral/influencer
# users convert and retain better.
CHANNELS = {
    #               share, quality_mult
    "referral":       (0.22, 1.30),
    "organic":        (0.20, 1.10),
    "influencer":     (0.18, 1.20),
    "app_store":      (0.16, 1.00),
    "social_ads":     (0.24, 0.78),
}
chan_names = list(CHANNELS.keys())
chan_share = np.array([v[0] for v in CHANNELS.values()])
chan_share = chan_share / chan_share.sum()
chan_quality = {k: v[1] for k, v in CHANNELS.items()}

DEVICES = (["Android", "iOS", "Web"], [0.58, 0.30, 0.12])      # India-leaning mix
REGIONS = (["Metro", "Tier 1", "Tier 2", "Tier 3"], [0.34, 0.28, 0.24, 0.14])
AGE_GROUPS = (["18-24", "25-34", "35-44", "45+"], [0.42, 0.34, 0.16, 0.08])

# Hour-of-day weights (0-23): overnight trough, lunch bump, big evening peak.
HOUR_WEIGHTS = np.array([
    0.6, 0.4, 0.3, 0.3, 0.4, 0.6,   # 0-5  overnight
    1.0, 1.6, 2.0, 1.8, 1.6, 1.8,   # 6-11 morning ramp
    2.6, 2.4, 1.8, 1.6, 1.8, 2.2,   # 12-17 lunch bump + afternoon
    3.0, 4.2, 5.0, 4.6, 3.4, 1.6,   # 18-23 evening peak
])
HOUR_WEIGHTS = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()

# Day-of-week multiplier (Mon=0...Sun=6): weekends a bit higher.
DOW_MULT = np.array([1.00, 0.98, 1.00, 1.02, 1.10, 1.22, 1.18])

1. Users + funnel

In [4]:
def pick(choices_weights, n):
    choices, weights = choices_weights
    return rng.choice(choices, size=n, p=np.array(weights) / np.sum(weights))

user_id = np.array([f"U{100000 + i}" for i in range(N_USERS)])

# Waitlist signup date — slightly front-loaded (launch announcements spike early).
signup_offset = (rng.beta(1.6, 2.2, N_USERS) * WAITLIST_DAYS).astype(int)
waitlist_signup_date = np.array(
    [WAITLIST_START + timedelta(days=int(d)) for d in signup_offset]
)

channel = rng.choice(chan_names, size=N_USERS, p=chan_share)
device = pick(DEVICES, N_USERS)
region = pick(REGIONS, N_USERS)
age_group = pick(AGE_GROUPS, N_USERS)

# Latent engagement propensity (0-1): the user's intrinsic likelihood to engage.
# Beta distribution skewed toward low-medium (most signups are lukewarm),
# nudged up/down by acquisition-channel quality.
base_prop = rng.beta(2.0, 3.2, N_USERS)
qual = np.array([chan_quality[c] for c in channel])
engagement_propensity = np.clip(base_prop * (0.7 + 0.3 * qual), 0, 1)

# --- Funnel stage 1: waitlist -> onboarding completed ---
# Base ~62%, scaled by propensity and channel quality.
p_onboard = np.clip(0.30 + 0.55 * engagement_propensity, 0.05, 0.95)
onboarding_completed = rng.random(N_USERS) < p_onboard

# --- Funnel stage 2: onboarding -> first opinion posted ---
# THE classic social-app leak. Only onboarded users can post.
p_first_post = np.clip(0.20 + 0.65 * engagement_propensity, 0.03, 0.92)
first_opinion_posted = onboarding_completed & (rng.random(N_USERS) < p_first_post)

# --- Funnel stage 3: first opinion -> week-1 retention ---
# Posting strongly predicts retention; onboarded-but-never-posted (browsers)
# retain much worse.
p_ret_poster = np.clip(0.35 + 0.55 * engagement_propensity, 0.05, 0.95)
p_ret_browser = np.clip(0.08 + 0.25 * engagement_propensity, 0.02, 0.60)
rand_ret = rng.random(N_USERS)
active_week_1 = np.where(
    first_opinion_posted,
    rand_ret < p_ret_poster,
    onboarding_completed & (rand_ret < p_ret_browser),  # browsers can still return
)

# Stage dates (only where the stage happened)
def stage_date(base_dates, happened, lo, hi):
    out = np.array([pd.NaT] * N_USERS, dtype="object")
    gap = rng.integers(lo, hi + 1, N_USERS)
    for i in range(N_USERS):
        if happened[i]:
            out[i] = base_dates[i] + timedelta(days=int(gap[i]))
    return out

onboarding_date = stage_date(waitlist_signup_date, onboarding_completed, 0, 14)
# first post happens after onboarding
first_post_date = np.array([pd.NaT] * N_USERS, dtype="object")
gap_post = rng.integers(0, 8, N_USERS)
for i in range(N_USERS):
    if first_opinion_posted[i] and onboarding_date[i] is not pd.NaT:
        first_post_date[i] = onboarding_date[i] + timedelta(days=int(gap_post[i]))

# Behavioural segment (derived, useful for the dashboard)
def segment(i):
    if not onboarding_completed[i]:
        return "Dropped (no onboarding)"
    if not first_opinion_posted[i]:
        return "Browser"
    if engagement_propensity[i] > 0.62:
        return "Creator"
    return "Voter"

user_segment = np.array([segment(i) for i in range(N_USERS)])

users = pd.DataFrame({
    "user_id": user_id,
    "waitlist_signup_date": pd.to_datetime(waitlist_signup_date),
    "acquisition_channel": channel,
    "device": device,
    "region": region,
    "age_group": age_group,
    "onboarding_completed": onboarding_completed,
    "onboarding_date": pd.to_datetime(onboarding_date),
    "first_opinion_posted": first_opinion_posted,
    "first_post_date": pd.to_datetime(first_post_date),
    "active_week_1": active_week_1,
    "user_segment": user_segment,
    "engagement_propensity": np.round(engagement_propensity, 3),
})


2. Engagement events (simulated 8-week beta)

In [6]:
#Events are generated for users who completed onboarding. Posters (voters/
# creators) generate many actions incl. 'pick'; browsers generate mostly 'browse'.
event_rows = []
beta_seconds = BETA_DAYS * 24 * 3600

onboarded_idx = np.where(onboarding_completed)[0]

ACTIONS_POSTER = (["pick", "browse", "react", "comment", "share"],
                  [0.34, 0.34, 0.18, 0.09, 0.05])
ACTIONS_BROWSER = (["browse", "react", "pick"],
                   [0.86, 0.11, 0.03])

eid = 0
for i in onboarded_idx:
    prop = engagement_propensity[i]
    is_poster = first_opinion_posted[i]
    retained = active_week_1[i]

    # Number of events scales with propensity, posting, and retention.
    if is_poster:
        lam = 6 + 34 * prop + (10 if retained else 0)
    else:
        lam = 2 + 8 * prop + (4 if retained else 0)
    n_ev = int(rng.poisson(max(lam, 1)))
    if n_ev == 0:
        continue

    actions_cw = ACTIONS_POSTER if is_poster else ACTIONS_BROWSER
    acts = rng.choice(actions_cw[0], size=n_ev,
                      p=np.array(actions_cw[1]) / np.sum(actions_cw[1]))

    # Each user favours a couple of categories (interest concentration).
    fav = rng.choice(cat_names, size=2, replace=False, p=cat_share)
    user_cat_p = cat_share.copy()
    for f in fav:
        user_cat_p[cat_names.index(f)] *= 3.0
    user_cat_p = user_cat_p / user_cat_p.sum()
    cats = rng.choice(cat_names, size=n_ev, p=user_cat_p)

    # Activity window: starts around onboarding, spread across beta.
    start_anchor = max(0, (pd.to_datetime(onboarding_date[i]) - BETA_START).days) \
        if onboarding_date[i] is not pd.NaT else 0
    start_anchor = min(start_anchor, BETA_DAYS - 1)

    # Sessions: cluster events into a few sessions.
    n_sessions = max(1, int(np.ceil(n_ev / rng.integers(3, 7))))
    session_days = np.sort(rng.integers(start_anchor, BETA_DAYS, n_sessions))

    for k in range(n_ev):
        day = int(rng.choice(session_days))
        # DOW-weighted nudge
        cur_date = BETA_START + timedelta(days=day)
        dow = cur_date.weekday()
        hour = int(rng.choice(np.arange(24),
                              p=HOUR_WEIGHTS * DOW_MULT[dow] /
                                np.sum(HOUR_WEIGHTS * DOW_MULT[dow])))
        minute = int(rng.integers(0, 60))
        second = int(rng.integers(0, 60))
        ts = cur_date + timedelta(hours=hour, minutes=minute, seconds=second)

        cat = cats[k]
        act = acts[k]
        # Reactions received only meaningful for content actions.
        if act in ("pick", "comment"):
            base = rng.poisson(4 * cat_eng[cat] * (0.6 + prop))
            # evening peak gets a small virality boost
            if 19 <= hour <= 22:
                base = int(base * 1.25)
            reactions = int(base)
        else:
            reactions = 0

        session_id = f"S{i}_{day}"
        event_rows.append((
            f"E{1000000 + eid}", user_id[i], ts, ts.date().isoformat(),
            hour, cur_date.strftime("%A"), cat, act, reactions, session_id,
        ))
        eid += 1

events = pd.DataFrame(event_rows, columns=[
    "event_id", "user_id", "event_timestamp", "event_date", "event_hour",
    "day_of_week", "category", "action_type", "reactions_received", "session_id",
])
events["event_timestamp"] = pd.to_datetime(events["event_timestamp"])
events = events.sort_values("event_timestamp").reset_index(drop=True)


3. Tidy funnel summary (pre-aggregated for Power BI funnel visual)

In [7]:
n_wait = len(users)
n_onb = int(users["onboarding_completed"].sum())
n_post = int(users["first_opinion_posted"].sum())
n_ret = int(users["active_week_1"].sum())

funnel = pd.DataFrame({
    "stage_order": [1, 2, 3, 4],
    "funnel_stage": ["Waitlist signup", "Onboarding completed",
                     "First opinion posted", "Week-1 retained"],
    "users": [n_wait, n_onb, n_post, n_ret],
})
funnel["pct_of_top"] = (funnel["users"] / n_wait * 100).round(1)
funnel["step_conversion_pct"] = (
    funnel["users"] / funnel["users"].shift(1) * 100
).round(1)
funnel.loc[0, "step_conversion_pct"] = 100.0


4. Data dictionary

In [8]:
dictionary = pd.DataFrame([
    ("pickspool_users", "user_id", "Unique simulated user / waitlist signup ID"),
    ("pickspool_users", "waitlist_signup_date", "Date user joined the waitlist"),
    ("pickspool_users", "acquisition_channel", "referral / organic / influencer / app_store / social_ads"),
    ("pickspool_users", "device", "Android / iOS / Web"),
    ("pickspool_users", "region", "Metro / Tier 1 / Tier 2 / Tier 3"),
    ("pickspool_users", "age_group", "18-24 / 25-34 / 35-44 / 45+"),
    ("pickspool_users", "onboarding_completed", "TRUE if user finished onboarding (funnel stage 2)"),
    ("pickspool_users", "onboarding_date", "Date onboarding completed (blank if not)"),
    ("pickspool_users", "first_opinion_posted", "TRUE if user posted their first opinion/pick (stage 3)"),
    ("pickspool_users", "first_post_date", "Date of first opinion posted (blank if not)"),
    ("pickspool_users", "active_week_1", "TRUE if user active in their first week (stage 4 / retention)"),
    ("pickspool_users", "user_segment", "Derived: Dropped / Browser / Voter / Creator"),
    ("pickspool_users", "engagement_propensity", "Latent 0-1 simulated engagement score (model input; usually internal)"),
    ("pickspool_events", "event_id", "Unique event ID"),
    ("pickspool_events", "user_id", "FK -> pickspool_users.user_id"),
    ("pickspool_events", "event_timestamp", "Full timestamp of the action"),
    ("pickspool_events", "event_date", "Date of the action (for date hierarchy)"),
    ("pickspool_events", "event_hour", "Hour of day 0-23 (posting-time analysis)"),
    ("pickspool_events", "day_of_week", "Monday..Sunday"),
    ("pickspool_events", "category", "Opinion category the action belongs to"),
    ("pickspool_events", "action_type", "pick (cast opinion) / browse / react / comment / share"),
    ("pickspool_events", "reactions_received", "Reactions a pick/comment earned (engagement proxy)"),
    ("pickspool_events", "session_id", "Groups events into sessions"),
    ("pickspool_funnel_stages", "funnel_stage", "Named funnel stage"),
    ("pickspool_funnel_stages", "users", "Count of users reaching the stage"),
    ("pickspool_funnel_stages", "pct_of_top", "% of original waitlist reaching the stage"),
    ("pickspool_funnel_stages", "step_conversion_pct", "Conversion from the previous stage"),
], columns=["table", "column", "description"])


Write outputs

In [9]:
out = "data"
os.makedirs(out, exist_ok=True)
users.to_csv(f"{out}/pickspool_users.csv", index=False)
events.to_csv(f"{out}/pickspool_events.csv", index=False)
funnel.to_csv(f"{out}/pickspool_funnel_stages.csv", index=False)
dictionary.to_csv(f"{out}/data_dictionary.csv", index=False)

print("=== GENERATION COMPLETE ===")
print(f"users rows : {len(users):,}")
print(f"events rows: {len(events):,}")
print(f"funnel     : waitlist {n_wait:,} -> onboard {n_onb:,} -> "
      f"post {n_post:,} -> retained {n_ret:,}")
print(f"overall waitlist->retained: {n_ret/n_wait*100:.1f}%")
print("\nfunnel table:")
print(funnel.to_string(index=False))

=== GENERATION COMPLETE ===
users rows : 5,000
events rows: 42,523
funnel     : waitlist 5,000 -> onboard 2,532 -> post 1,236 -> retained 972
overall waitlist->retained: 19.4%

funnel table:
 stage_order         funnel_stage  users  pct_of_top  step_conversion_pct
           1      Waitlist signup   5000       100.0                100.0
           2 Onboarding completed   2532        50.6                 50.6
           3 First opinion posted   1236        24.7                 48.8
           4      Week-1 retained    972        19.4                 78.6


In [10]:
!zip -r pickspool_data.zip data charts
from google.colab import files
files.download('pickspool_data.zip')

	zip warning: name not matched: charts
  adding: data/ (stored 0%)
  adding: data/data_dictionary.csv (deflated 60%)
  adding: data/pickspool_events.csv (deflated 81%)
  adding: data/pickspool_users.csv (deflated 87%)
  adding: data/pickspool_funnel_stages.csv (deflated 23%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import os
D = "data"
C = "charts"
os.makedirs(C, exist_ok=True)
users = pd.read_csv(f"{D}/pickspool_users.csv", parse_dates=[
    "waitlist_signup_date", "onboarding_date", "first_post_date"])
events = pd.read_csv(f"{D}/pickspool_events.csv", parse_dates=["event_timestamp"])
funnel = pd.read_csv(f"{D}/pickspool_funnel_stages.csv")
INK = "#1b2330"; ACCENT = "#3b6ef0"; ACCENT2 = "#16b5a0"; WARN = "#ef5b5b"
GREY = "#aeb6c2"
plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.edgecolor": "#d6dbe3", "axes.titleweight": "bold"})

print("=" * 64)
print("HEADLINE INSIGHTS  (synthetic — framework demo)")
print("=" * 64)

HEADLINE INSIGHTS  (synthetic — framework demo)


# 1. Funnel + biggest leak



In [13]:
biggest_leak = funnel.iloc[1:]["step_conversion_pct"].idxmin()
leak_row = funnel.loc[biggest_leak]
print(f"\n1) FUNNEL  waitlist->retained = {funnel.iloc[-1]['pct_of_top']}% of signups")
print(f"   Biggest single leak: {funnel.loc[biggest_leak-1,'funnel_stage']} -> "
      f"{leak_row['funnel_stage']} ({leak_row['step_conversion_pct']}% pass)")



1) FUNNEL  waitlist->retained = 19.4% of signups
   Biggest single leak: Onboarding completed -> First opinion posted (48.8% pass)


# 2. Channel quality

In [14]:
ch = users.groupby("acquisition_channel").agg(
    signups=("user_id", "size"),
    activation=("first_opinion_posted", "mean"),
    retention=("active_week_1", "mean")).sort_values("retention", ascending=False)
ch["activation"] = (ch["activation"] * 100).round(1)
ch["retention"] = (ch["retention"] * 100).round(1)
print("\n2) CHANNEL QUALITY (activation% / retention%)")
print(ch.to_string())


2) CHANNEL QUALITY (activation% / retention%)
                     signups  activation  retention
acquisition_channel                                
referral                1098        29.9       23.1
organic                 1013        24.0       19.1
influencer               896        25.6       18.6
social_ads              1220        21.3       18.6
app_store                773        22.8       16.9


## 3. Voter vs Browser retention

In [15]:
seg = users[users["onboarding_completed"]].copy()
seg["grp"] = np.where(seg["first_opinion_posted"], "Voter/Creator (posted)", "Browser (never posted)")
seg_ret = seg.groupby("grp")["active_week_1"].mean() * 100
print("\n3) RETENTION: voters vs browsers")
print(seg_ret.round(1).to_string())
lift = seg_ret["Voter/Creator (posted)"] / seg_ret["Browser (never posted)"]
print(f"   -> posting users retain {lift:.1f}x better than browsers")


3) RETENTION: voters vs browsers
grp
Browser (never posted)    16.2
Voter/Creator (posted)    61.7
   -> posting users retain 3.8x better than browsers


# 4. Category engagement

In [16]:
cat = events[events["action_type"] == "pick"].groupby("category").agg(
    picks=("event_id", "size"),
    avg_reactions=("reactions_received", "mean")).sort_values("picks", ascending=False)
cat["avg_reactions"] = cat["avg_reactions"].round(2)
print("\n4) TOP CATEGORIES by picks cast")
print(cat.head(6).to_string())

# 5. Posting-time peak
hourly = events.groupby("event_hour").size()
peak_hour = hourly.idxmax()
print(f"\n5) PEAK POSTING WINDOW: {peak_hour:02d}:00-{peak_hour+1:02d}:00 "
      f"({hourly.max():,} events). Evening 19-22h carries "
      f"{hourly.loc[19:22].sum()/hourly.sum()*100:.0f}% of all activity.")


4) TOP CATEGORIES by picks cast
                picks  avg_reactions
category                            
Sports           2124           6.65
Movies & TV      1852           6.08
Memes & Humour   1755           7.05
Music            1381           6.02
Food & Drink     1241           5.00
Gaming           1024           5.55

5) PEAK POSTING WINDOW: 20:00-21:00 (4,494 events). Evening 19-22h carries 37% of all activity.


# CHARTS

Chart 1: Funnel

In [17]:
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(funnel))[::-1]
bars = ax.barh(y, funnel["users"], color=[ACCENT, ACCENT, WARN, ACCENT2],
               edgecolor="white")
for i, (yy, row) in enumerate(zip(y, funnel.itertuples())):
    ax.text(row.users + 60, yy, f"{row.users:,}  ({row.pct_of_top}%)",
            va="center", fontsize=10, color=INK)
ax.set_yticks(y); ax.set_yticklabels(funnel["funnel_stage"])
ax.set_title("Pickspool waitlist-to-active funnel (simulated)")
ax.set_xlabel("Users"); ax.set_xlim(0, funnel["users"].max() * 1.25)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.text(0.99, -0.18, "Synthetic data — industry-benchmarked patterns",
        transform=ax.transAxes, ha="right", fontsize=8, color=GREY)
plt.tight_layout(); plt.savefig(f"{C}/01_funnel.png"); plt.close()

Chart 2: Channel activation vs retention

In [18]:
fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(ch)); w = 0.4
ax.bar(x - w/2, ch["activation"], w, label="Activation %", color=ACCENT)
ax.bar(x + w/2, ch["retention"], w, label="Wk-1 retention %", color=ACCENT2)
ax.set_xticks(x); ax.set_xticklabels(ch.index, rotation=15)
ax.set_title("Activation & retention by acquisition channel (simulated)")
ax.set_ylabel("%"); ax.legend(frameon=False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/02_channels.png"); plt.close()

Chart 3: Voter vs browser retention

In [19]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
g = seg_ret.sort_values()
ax.bar(g.index, g.values, color=[GREY, ACCENT], edgecolor="white")
for i, v in enumerate(g.values):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", fontweight="bold", color=INK)
ax.set_title("Week-1 retention: posting vs browsing (simulated)")
ax.set_ylabel("Retention %"); ax.set_ylim(0, max(g.values) * 1.2)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/03_voter_vs_browser.png"); plt.close()

Chart 4: Category picks

In [20]:
fig, ax = plt.subplots(figsize=(8, 4.6))
cc = cat.sort_values("picks")
ax.barh(cc.index, cc["picks"], color=ACCENT, edgecolor="white")
ax.set_title("Opinions cast by category (simulated)")
ax.set_xlabel("Picks cast")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/04_categories.png"); plt.close()

Chart 5: Hourly posting pattern

In [21]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(hourly.index, hourly.values, color=ACCENT, alpha=0.25)
ax.plot(hourly.index, hourly.values, color=ACCENT, linewidth=2)
ax.axvspan(19, 22, color=WARN, alpha=0.12)
ax.set_title("Engagement by hour of day (simulated)")
ax.set_xlabel("Hour"); ax.set_ylabel("Events"); ax.set_xticks(range(0, 24, 2))
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/05_hourly.png"); plt.close()

print("\nCharts saved to /charts (5 PNGs).")


Charts saved to /charts (5 PNGs).



Pickspool Growth Analytics
=====================================
Two interpretable, analyst-appropriate models on the synthetic data:

  MODEL 1  
  Logistic Regression->
  predict week-1 retention from DAY-1 behaviour
           (an "early-warning" score the team could act on after a user's first
            day). Output = interpretable odds ratios + a per-user retention score.

  MODEL 2  
  K-Means clustering-> behavioural user segmentation into personas,
           mirroring the persona work in the A/B-testing project.

Honesty note: data is synthetic. The latent `engagement_propensity` (the hidden
generator field) is deliberately EXCLUDED from every model — using it would be
leakage. Models learn only from observable signup + early-activity features.

In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, accuracy_score, silhouette_score

import os
SEED = 42
D = "data"
C = "charts"
os.makedirs(C, exist_ok=True)
INK, ACCENT, ACCENT2, WARN, GREY = "#1b2330", "#3b6ef0", "#16b5a0", "#ef5b5b", "#aeb6c2"
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.titleweight": "bold"})
users = pd.read_csv(f"{D}/pickspool_users.csv",
                    parse_dates=["waitlist_signup_date", "onboarding_date", "first_post_date"])
events = pd.read_csv(f"{D}/pickspool_events.csv", parse_dates=["event_timestamp"])

# FEATURE ENGINEERING — each user's first active DAY (no target leakage)

In [25]:
events["event_day"] = events["event_timestamp"].dt.normalize()
first_day = events.groupby("user_id")["event_day"].transform("min")
day1 = events[events["event_day"] == first_day].copy()

day1_feats = day1.groupby("user_id").agg(
    d1_events=("event_id", "size"),
    d1_picks=("action_type", lambda s: (s == "pick").sum()),
    d1_categories=("category", "nunique"),
    d1_reactions=("reactions_received", "sum"),
).reset_index()
# Model on the actionable cohort: users who onboarded
df = users[users["onboarding_completed"]].merge(day1_feats, on="user_id", how="left")
for col in ["d1_events", "d1_picks", "d1_categories", "d1_reactions"]:
    df[col] = df[col].fillna(0)
CAT = ["acquisition_channel", "device", "region", "age_group"]
NUM = ["d1_events", "d1_picks", "d1_categories", "d1_reactions"]
y = df["active_week_1"].astype(int)
X = df[CAT + NUM]

# MODEL 1 — Logistic Regression (retention early-warning)

In [26]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT),
    ("num", StandardScaler(), NUM),
])
clf = Pipeline([("pre", pre),
                ("lr", LogisticRegression(max_iter=1000, class_weight="balanced"))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
clf.fit(Xtr, ytr)
proba_te = clf.predict_proba(Xte)[:, 1]
pred_te = (proba_te >= 0.5).astype(int)
auc = roc_auc_score(yte, proba_te)
acc = accuracy_score(yte, pred_te)
base = yte.mean()
print("=" * 64)
print("MODEL 1 — Logistic Regression: predict week-1 retention from day-1")
print("=" * 64)
print(f"Cohort (onboarded users): {len(df):,}  |  base retention rate: {y.mean()*100:.1f}%")
print(f"Test ROC-AUC: {auc:.3f}   Accuracy: {acc:.3f}   (baseline guess-all: {max(base,1-base)*100:.1f}%)")
# Interpretable odds ratios
feat_names = (clf.named_steps["pre"].named_transformers_["cat"]
              .get_feature_names_out(CAT).tolist()) + NUM
coefs = clf.named_steps["lr"].coef_[0]
odds = pd.DataFrame({"feature": feat_names, "coef": coefs,
                     "odds_ratio": np.exp(coefs)}).sort_values("odds_ratio", ascending=False)
print("\nTop drivers of retention (odds ratio > 1 = more likely to retain):")
print(odds.head(6).to_string(index=False))
print("\nTop drag factors (odds ratio < 1 = less likely to retain):")
print(odds.tail(4).to_string(index=False))

# Score EVERY onboarded user (the deliverable for Power BI)
df["predicted_retention_prob"] = clf.predict_proba(X)[:, 1].round(3)

# Chart: odds ratios
fig, ax = plt.subplots(figsize=(8, 5))
top = pd.concat([odds.head(6), odds.tail(4)]).drop_duplicates("feature")
colors = [ACCENT2 if v > 1 else WARN for v in top["odds_ratio"]]
ax.barh(top["feature"][::-1], top["odds_ratio"][::-1], color=colors[::-1], edgecolor="white")
ax.axvline(1.0, color=INK, lw=1, ls="--")
ax.set_title("What drives week-1 retention — odds ratios (simulated)")
ax.set_xlabel("Odds ratio (>1 helps retention, <1 hurts)")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.text(0.99, -0.16, "Logistic regression on synthetic day-1 features",
        transform=ax.transAxes, ha="right", fontsize=8, color=GREY)
plt.tight_layout(); plt.savefig(f"{C}/06_retention_drivers.png"); plt.close()


MODEL 1 — Logistic Regression: predict week-1 retention from day-1
Cohort (onboarded users): 2,532  |  base retention rate: 38.4%
Test ROC-AUC: 0.659   Accuracy: 0.645   (baseline guess-all: 61.6%)

Top drivers of retention (odds ratio > 1 = more likely to retain):
                       feature     coef  odds_ratio
                  d1_reactions 0.752730    2.122787
  acquisition_channel_referral 0.447457    1.564329
acquisition_channel_social_ads 0.379020    1.460852
                 region_Tier 3 0.264311    1.302533
                 region_Tier 2 0.236414    1.266698
   acquisition_channel_organic 0.219142    1.245008

Top drag factors (odds ratio < 1 = less likely to retain):
        feature      coef  odds_ratio
     device_Web -0.066628    0.935543
age_group_25-34 -0.091459    0.912598
  age_group_45+ -0.238628    0.787708
age_group_35-44 -0.246991    0.781147


# MODEL 2 — K-Means behavioural segmentation

In [27]:
print("\n" + "=" * 64)
print("MODEL 2 — K-Means: behavioural user segmentation")
print("=" * 64)

# Behaviour profile per active user (anyone with >=1 event)
prof = events.groupby("user_id").agg(
    total_events=("event_id", "size"),
    total_picks=("action_type", lambda s: (s == "pick").sum()),
    distinct_categories=("category", "nunique"),
    total_reactions=("reactions_received", "sum"),
    num_sessions=("session_id", "nunique"),
).reset_index()
prof["avg_reactions_per_event"] = (prof["total_reactions"] / prof["total_events"]).round(2)

KFEATS = ["total_events", "total_picks", "distinct_categories",
          "total_reactions", "num_sessions"]
Xk = StandardScaler().fit_transform(prof[KFEATS])

# Pick k via elbow + silhouette
inertias, sils = [], []
Ks = range(2, 8)
for k in Ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(Xk)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xk, km.labels_))
best_k = 4  # chosen: clean elbow + strong, interpretable personas
print(f"Silhouette by k: " + ", ".join(f"{k}:{s:.3f}" for k, s in zip(Ks, sils)))
print(f"Chosen k = {best_k}")

km = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit(Xk)
prof["cluster"] = km.labels_

# Profile + auto-label clusters by behaviour
cluster_profile = prof.groupby("cluster")[KFEATS].mean().round(1)
cluster_profile["users"] = prof.groupby("cluster").size()
# Rank clusters by overall activity to assign sensible names
order = cluster_profile["total_events"].sort_values(ascending=False).index.tolist()
label_map = {}
names = ["Power creators", "Active voters", "Casual dippers", "Low-touch browsers"]
for rank, cl in enumerate(order):
    label_map[cl] = names[rank] if rank < len(names) else f"Segment {cl}"
prof["cluster_label"] = prof["cluster"].map(label_map)
cluster_profile["label"] = [label_map[i] for i in cluster_profile.index]
print("\nCluster profiles (means):")
print(cluster_profile.to_string())
# Chart: cluster sizes + picks
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
cp = cluster_profile.sort_values("total_events", ascending=False)
axes[0].bar(cp["label"], cp["users"], color=ACCENT, edgecolor="white")
axes[0].set_title("Segment sizes (simulated)"); axes[0].set_ylabel("Users")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(cp["label"], cp["total_picks"], color=ACCENT2, edgecolor="white")
axes[1].set_title("Avg picks cast per segment (simulated)"); axes[1].set_ylabel("Avg picks")
axes[1].tick_params(axis="x", rotation=20)
for ax in axes:
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/07_segments.png"); plt.close()
# Elbow chart
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(list(Ks), inertias, "-o", color=ACCENT)
ax.axvline(best_k, color=WARN, ls="--", lw=1)
ax.set_title("K-Means elbow (simulated)"); ax.set_xlabel("k"); ax.set_ylabel("Inertia")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{C}/08_elbow.png"); plt.close()



MODEL 2 — K-Means: behavioural user segmentation
Silhouette by k: 2:0.624, 3:0.543, 4:0.415, 5:0.388, 6:0.361, 7:0.355
Chosen k = 4

Cluster profiles (means):
         total_events  total_picks  distinct_categories  total_reactions  num_sessions  users               label
cluster                                                                                                          
0                10.4          1.4                  6.1              7.3           2.8    570      Casual dippers
1                25.6          8.4                  8.7             55.8           5.9    637       Active voters
2                39.5         14.1                 10.0            114.1           9.1    422      Power creators
3                 4.1          0.2                  3.0              0.7           1.3    882  Low-touch browsers


# ENRICHED USERS FILE FOR POWER BI (cluster label + retention score)

In [28]:
enriched = users.merge(
    df[["user_id", "predicted_retention_prob"]], on="user_id", how="left"
).merge(
    prof[["user_id", "cluster_label"]], on="user_id", how="left"
)
enriched["cluster_label"] = enriched["cluster_label"].fillna("Did not activate")
enriched.to_csv(f"{D}/pickspool_users_enriched.csv", index=False)

print("\nSaved: data/pickspool_users_enriched.csv "
      "(adds predicted_retention_prob + cluster_label for Power BI slicers)")
print("Saved charts: 06_retention_drivers.png, 07_segments.png, 08_elbow.png")



Saved: data/pickspool_users_enriched.csv (adds predicted_retention_prob + cluster_label for Power BI slicers)
Saved charts: 06_retention_drivers.png, 07_segments.png, 08_elbow.png
